In [1]:
# conda env: openmmlab_TS (prob can use mmdeploy just need albumentations)

In [2]:
import os

os.environ["CUDA_DEVICE_ORDER"]="PCI_BUS_ID"   # see issue #152
os.environ["CUDA_VISIBLE_DEVICES"]="0"

from pathlib import Path
import sys
from datetime import datetime

In [3]:
# which pretrained model to use (point to .pth file). Pretrained model should be the same model architecture. 
pretrained_model = Path("/n/groups/datta/6cam_keypoint_networks/mm_pose/Jonah/20241030_v1/rtmpose/rtmpose-m_8xb64-210e_ap10k-256x256_24-11-01-20-36-55/best_PCK_epoch_80.pth")
use_pretrained_model = True
output_directory = Path("/n/groups/datta/6cam_keypoint_networks/mm_pose/Jonah/20241216_v2")

# Where the COCO format dataset for finetuning is located (created in the previous notebook)
original_dataset_directory = Path("/n/groups/datta/6cam_keypoint_networks/training_data/JP_CW_scale_annos/COCO_format/")
new_dataset_directory = output_dir = Path("/n/groups/datta/6cam_keypoint_networks/training_data/combined_sets/JP_scale_and_forepaws/")


### shouldn't need to change below here for fine-tuning JP's 15 pt network ###

model_name = 'rtmpose-m_8xb64-210e_ap10k-256x256'

# this file contains info about the dataset (keypoints, skeleton, etc) needed for traiing
dataset_info_loc = Path("/n/groups/datta/Jonah/Local_code_groups/6cam_repos/multicam_airflow_pipeline/multicamera_airflow_pipeline/tim_240731/skeletons/weinreb15pt.py")
n_keypoints = 15

# which config to use (this is what we base the config off of). Should be in the mmpose repo. 
# config_loc = Path('/n/groups/datta/tim_sainburg/projects/mmpose/configs/animal_2d_keypoint/rtmpose/ap10k/rtmpose-m_8xb64-210e_ap10k-256x256.py')
config_loc = Path('/n/groups/datta/6cam_keypoint_networks/mm_pose/Jonah/20241216_v2/finetuning_config.py')

formatted_datetime = datetime.now().strftime("%y-%m-%d-%H-%M-%S")
working_directory = (output_directory / 'rtmpose' / f"{model_name}_{formatted_datetime}")
working_directory.mkdir(parents=True, exist_ok=True)

In [4]:
assert config_loc.exists()
assert new_dataset_directory.exists()
assert original_dataset_directory.exists()

In [5]:
from mmpose.registry import DATASETS
from mmpose.datasets.datasets.base import BaseCocoStyleDataset
dataset_type = 'CoCo15pt'  # must match class name
@DATASETS.register_module()
class CoCo15pt(BaseCocoStyleDataset):
    METAINFO: dict = dict(from_file=dataset_info_loc)

### Display compute / environment info (for future reference)

In [6]:
# Check nvcc version
!nvcc -V
# Check GCC version
!gcc --version

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2023 NVIDIA Corporation
Built on Mon_Apr__3_17:16:06_PDT_2023
Cuda compilation tools, release 12.1, V12.1.105
Build cuda_12.1.r12.1/compiler.32688072_0
gcc (GCC) 6.2.0
Copyright (C) 2016 Free Software Foundation, Inc.
This is free software; see the source for copying conditions.  There is NO
warranty; not even for MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.



In [7]:
from mmengine.utils import get_git_hash
from mmengine.utils.dl_utils import collect_env as collect_base_env
import sys
import mmdet
import torch
import torchvision
import mmpose
from mmcv.ops import get_compiling_cuda_version, get_compiler_version

def collect_env():
    """Collect the information of the running environments."""
    env_info = collect_base_env()
    env_info['MMDetection'] = f'{mmdet.__version__}+{get_git_hash()[:7]}'
    return env_info

print(f"Environment: {sys.executable}")
for name, val in collect_env().items():
    print(f'{name}: {val}')
# Check Pytorch installation
print('cuda version:', get_compiling_cuda_version())
print('compiler information:', get_compiler_version())
print('torch version:', torch.__version__, torch.cuda.is_available())
print('torchvision version:', torchvision.__version__)
print('mmpose version:', mmpose.__version__) 

Environment: /n/groups/datta/tim_sainburg/conda_envs/openmmlab/bin/python
sys.platform: linux
Python: 3.10.13 (main, Sep 11 2023, 13:44:35) [GCC 11.2.0]
CUDA available: True
numpy_random_seed: 2147483648
GPU 0: NVIDIA A100 80GB PCIe MIG 3g.40gb
CUDA_HOME: /n/groups/datta/tim_sainburg/conda_envs/openmmlab
NVCC: Cuda compilation tools, release 12.1, V12.1.105
GCC: gcc (GCC) 6.2.0
PyTorch: 2.1.0
PyTorch compiling details: PyTorch built with:
  - GCC 9.3
  - C++ Version: 201703
  - Intel(R) oneAPI Math Kernel Library Version 2023.1-Product Build 20230303 for Intel(R) 64 architecture applications
  - Intel(R) MKL-DNN v3.1.1 (Git Hash 64f6bcbcbab628e96f33a62c3e975f8535a7bde4)
  - OpenMP 201511 (a.k.a. OpenMP 4.5)
  - LAPACK is enabled (usually provided by MKL)
  - NNPACK is enabled
  - CPU capability usage: AVX512
  - CUDA Runtime 12.1
  - NVCC architecture flags: -gencode;arch=compute_50,code=sm_50;-gencode;arch=compute_60,code=sm_60;-gencode;arch=compute_61,code=sm_61;-gencode;arch=compute

### Create config file

In [8]:
from mmengine import Config

In [9]:
cfg = Config.fromfile(config_loc.as_posix())

In [10]:
# load COCO pre-trained weight
if use_pretrained_model:
    cfg.load_from = pretrained_model.as_posix()

In [11]:
# set the dataset directory
cfg.data_root = new_dataset_directory.as_posix()

# set the working directory
cfg.work_dir = working_directory.as_posix()
cfg.randomness = dict(seed=0)

In [12]:
# set dataset configs
cfg.dataset_type = dataset_type
cfg.data_mode = 'topdown'

# number of keypoints
cfg.model.head.out_channels = n_keypoints

cfg.train_dataloader.dataset.type = cfg.dataset_type
cfg.train_dataloader.dataset.ann_file = 'annotations/instances_train.json'
cfg.train_dataloader.dataset.data_root = cfg.data_root
cfg.train_dataloader.dataset.data_prefix = dict(img='train/')


cfg.val_dataloader.dataset.type = cfg.dataset_type
cfg.val_dataloader.dataset.bbox_file = None
cfg.val_dataloader.dataset.ann_file = 'annotations/instances_val.json'
cfg.val_dataloader.dataset.data_root = cfg.data_root
cfg.val_dataloader.dataset.data_prefix = dict(img='val/')

cfg.test_dataloader.dataset.type = cfg.dataset_type
cfg.test_dataloader.dataset.bbox_file = None
cfg.test_dataloader.dataset.ann_file = 'annotations/instances_val.json'
cfg.test_dataloader.dataset.data_root = cfg.data_root
cfg.test_dataloader.dataset.data_prefix = dict(img='val/')

# set to custom datset
cfg.train_dataloader.dataset.metainfo = dict(from_file=dataset_info_loc.as_posix())
cfg.val_dataloader.dataset.metainfo = dict(from_file=dataset_info_loc.as_posix())
cfg.test_dataloader.dataset.metainfo = dict(from_file=dataset_info_loc.as_posix())

# set evaluator
cfg.val_evaluator = dict(type='PCKAccuracy')
cfg.test_evaluator = cfg.val_evaluator

cfg.default_hooks.checkpoint.save_best = 'PCK'
cfg.default_hooks.checkpoint.max_keep_ckpts = 15
cfg.default_hooks.checkpoint.interval = 10

cfg.max_epochs = 2000
cfg.train_cfg.max_epochs = 2000

In [13]:
print(cfg)

Config (path: /n/groups/datta/6cam_keypoint_networks/mm_pose/Jonah/20241216_v2/finetuning_config.py): {'auto_scale_lr': {'base_batch_size': 512}, 'backend_args': {'backend': 'local'}, 'base_lr': 0.0005, 'codec': {'input_size': (256, 256), 'normalize': False, 'sigma': (5.66, 5.66), 'simcc_split_ratio': 2.0, 'type': 'SimCCLabel', 'use_dark': False}, 'custom_hooks': [{'ema_type': 'ExpMomentumEMA', 'momentum': 0.0002, 'priority': 49, 'type': 'EMAHook', 'update_buffers': True}, {'switch_epoch': 180, 'switch_pipeline': [{'backend_args': {'backend': 'local'}, 'type': 'LoadImage'}, {'type': 'GetBBoxCenterScale'}, {'direction': 'horizontal', 'type': 'RandomFlip'}, {'type': 'RandomHalfBody'}, {'rotate_factor': 60, 'scale_factor': [0.75, 1.25], 'shift_factor': 0.0, 'type': 'RandomBBoxTransform'}, {'input_size': (256, 256), 'type': 'TopdownAffine'}, {'type': 'mmdet.YOLOXHSVRandomAug'}, {'transforms': [{'p': 0.1, 'type': 'Blur'}, {'p': 0.1, 'type': 'MedianBlur'}, {'max_height': 0.4, 'max_holes': 1,

In [14]:
# set preprocess configs to model
cfg.model.setdefault('data_preprocessor', cfg.get('preprocess_cfg', {}))

{'bgr_to_rgb': True,
 'mean': [123.675, 116.28, 103.53],
 'std': [58.395, 57.12, 57.375],
 'type': 'PoseDataPreprocessor'}

In [15]:
# save configuration file for future reference
cfg.dump(working_directory / 'config.py')

### run network

In [16]:
from mmengine.config import Config, DictAction
from mmengine.runner import Runner

In [17]:
# build the runner from config
runner = Runner.from_cfg(cfg)

12/16 14:53:07 - mmengine - INFO - 
------------------------------------------------------------
System environment:
    sys.platform: linux
    Python: 3.10.13 (main, Sep 11 2023, 13:44:35) [GCC 11.2.0]
    CUDA available: True
    numpy_random_seed: 0
    GPU 0: NVIDIA A100 80GB PCIe MIG 3g.40gb
    CUDA_HOME: /n/groups/datta/tim_sainburg/conda_envs/openmmlab
    NVCC: Cuda compilation tools, release 12.1, V12.1.105
    GCC: gcc (GCC) 6.2.0
    PyTorch: 2.1.0
    PyTorch compiling details: PyTorch built with:
  - GCC 9.3
  - C++ Version: 201703
  - Intel(R) oneAPI Math Kernel Library Version 2023.1-Product Build 20230303 for Intel(R) 64 architecture applications
  - Intel(R) MKL-DNN v3.1.1 (Git Hash 64f6bcbcbab628e96f33a62c3e975f8535a7bde4)
  - OpenMP 201511 (a.k.a. OpenMP 4.5)
  - LAPACK is enabled (usually provided by MKL)
  - NNPACK is enabled
  - CPU capability usage: AVX512
  - CUDA Runtime 12.1
  - NVCC architecture flags: -gencode;arch=compute_50,code=sm_50;-gencode;arch=compu

In [18]:
# start training
runner.train()

/n/groups/datta/tim_sainburg/conda_envs/openmmlab/lib/python3.10/site-packages/mmpose/datasets/transforms/common_transforms.py:656: UserWarning: Blur is not pixel-level transformations. Please use with caution.
  warnings.warn(
/n/groups/datta/tim_sainburg/conda_envs/openmmlab/lib/python3.10/site-packages/mmpose/datasets/transforms/common_transforms.py:656: UserWarning: MedianBlur is not pixel-level transformations. Please use with caution.
  warnings.warn(
/n/groups/datta/tim_sainburg/conda_envs/openmmlab/lib/python3.10/site-packages/mmpose/datasets/transforms/common_transforms.py:656: UserWarning: CoarseDropout is not pixel-level transformations. Please use with caution.
  warnings.warn(


loading annotations into memory...
Done (t=0.13s)
creating index...
index created!
12/16 14:53:15 - mmengine - INFO - paramwise_options -- backbone.stem.0.bn.weight:weight_decay=0.0
12/16 14:53:15 - mmengine - INFO - paramwise_options -- backbone.stem.0.bn.bias:weight_decay=0.0
12/16 14:53:15 - mmengine - INFO - paramwise_options -- backbone.stem.1.bn.weight:weight_decay=0.0
12/16 14:53:15 - mmengine - INFO - paramwise_options -- backbone.stem.1.bn.bias:weight_decay=0.0
12/16 14:53:15 - mmengine - INFO - paramwise_options -- backbone.stem.2.bn.weight:weight_decay=0.0
12/16 14:53:15 - mmengine - INFO - paramwise_options -- backbone.stem.2.bn.bias:weight_decay=0.0
12/16 14:53:15 - mmengine - INFO - paramwise_options -- backbone.stage1.0.bn.weight:weight_decay=0.0
12/16 14:53:15 - mmengine - INFO - paramwise_options -- backbone.stage1.0.bn.bias:weight_decay=0.0
12/16 14:53:15 - mmengine - INFO - paramwise_options -- backbone.stage1.1.main_conv.bn.weight:weight_decay=0.0
12/16 14:53:15 - m

/n/groups/datta/tim_sainburg/conda_envs/openmmlab/lib/python3.10/site-packages/torch/utils/data/dataloader.py:557: UserWarning: This DataLoader will create 10 worker processes in total. Our suggested max number of worker in current system is 4, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(_create_warning_msg(


loading annotations into memory...
Done (t=0.01s)
creating index...
index created!
12/16 14:53:15 - mmengine - WARNING - The prefix is not set in metric class PCKAccuracy.
12/16 14:53:17 - mmengine - INFO - load backbone. in model from: https://download.openmmlab.com/mmpose/v1/projects/rtmposev1/cspnext-m_udp-aic-coco_210e-256x192-f2f7d6f6_20230130.pth
Loads checkpoint by http backend from path: https://download.openmmlab.com/mmpose/v1/projects/rtmposev1/cspnext-m_udp-aic-coco_210e-256x192-f2f7d6f6_20230130.pth
Loads checkpoint by local backend from path: /n/groups/datta/6cam_keypoint_networks/mm_pose/Jonah/20241030_v1/rtmpose/rtmpose-m_8xb64-210e_ap10k-256x256_24-11-01-20-36-55/best_PCK_epoch_80.pth
12/16 14:53:17 - mmengine - INFO - Load checkpoint from /n/groups/datta/6cam_keypoint_networks/mm_pose/Jonah/20241030_v1/rtmpose/rtmpose-m_8xb64-210e_ap10k-256x256_24-11-01-20-36-55/best_PCK_epoch_80.pth
12/16 14:53:17 - mmengine - WARNING - "FileClient" will be deprecated in future. Pleas

/n/groups/datta/tim_sainburg/conda_envs/openmmlab/lib/python3.10/site-packages/torch/utils/data/dataloader.py:557: UserWarning: This DataLoader will create 10 worker processes in total. Our suggested max number of worker in current system is 4, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(_create_warning_msg(


12/16 14:53:43 - mmengine - INFO - Epoch(train)    [1][ 50/262]  base_lr: 2.474750e-04 lr: 2.474750e-04  eta: 3 days, 5:09:07  time: 0.530104  data_time: 0.236119  memory: 5730  loss: 0.236671  loss_kpt: 0.236671  acc_pose: 0.814804
12/16 14:54:07 - mmengine - INFO - Epoch(train)    [1][100/262]  base_lr: 5.000000e-04 lr: 5.000000e-04  eta: 3 days, 1:28:30  time: 0.479671  data_time: 0.190632  memory: 5730  loss: 0.231701  loss_kpt: 0.231701  acc_pose: 0.852888
12/16 14:54:30 - mmengine - INFO - Epoch(train)    [1][150/262]  base_lr: 5.000000e-04 lr: 5.000000e-04  eta: 2 days, 23:24:04  time: 0.462276  data_time: 0.198227  memory: 5730  loss: 0.233787  loss_kpt: 0.233787  acc_pose: 0.864296
12/16 14:54:53 - mmengine - INFO - Epoch(train)    [1][200/262]  base_lr: 5.000000e-04 lr: 5.000000e-04  eta: 2 days, 22:05:03  time: 0.454665  data_time: 0.176000  memory: 5730  loss: 0.233158  loss_kpt: 0.233158  acc_pose: 0.879532
12/16 14:55:16 - mmengine - INFO - Epoch(train)    [1][250/262]  b

/n/groups/datta/tim_sainburg/conda_envs/openmmlab/lib/python3.10/site-packages/torch/utils/data/dataloader.py:557: UserWarning: This DataLoader will create 10 worker processes in total. Our suggested max number of worker in current system is 4, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(_create_warning_msg(


12/16 15:13:39 - mmengine - INFO - Epoch(val)   [10][50/58]    eta: 0:00:01  time: 0.223620  data_time: 0.111315  memory: 5730  
12/16 15:13:39 - mmengine - INFO - Evaluating PCKAccuracy (normalized by ``"bbox_size"``)...
12/16 15:13:39 - mmengine - INFO - Epoch(val) [10][58/58]    PCK: 0.927838  data_time: 0.099379  time: 0.204348
12/16 15:13:41 - mmengine - INFO - The best checkpoint with 0.9278 PCK at 10 epoch is saved to best_PCK_epoch_10.pth.
12/16 15:14:10 - mmengine - INFO - Epoch(train)   [11][ 50/262]  base_lr: 5.000000e-04 lr: 5.000000e-04  eta: 2 days, 18:56:49  time: 0.530891  data_time: 0.255300  memory: 5730  loss: 0.224313  loss_kpt: 0.224313  acc_pose: 0.915889
12/16 15:14:35 - mmengine - INFO - Epoch(train)   [11][100/262]  base_lr: 5.000000e-04 lr: 5.000000e-04  eta: 2 days, 18:59:41  time: 0.482643  data_time: 0.216900  memory: 5730  loss: 0.227486  loss_kpt: 0.227486  acc_pose: 0.883021
12/16 15:14:57 - mmengine - INFO - Epoch(train)   [11][150/262]  base_lr: 5.0000

### The config and path for running inference will be in the working directory